# STAT163 · Week 6 · Data formats

You can save the same table as different kinds of files, and each kind keeps some things
about the table and loses others. In this notebook we save one month of the online shop's
sales as a CSV file, an Excel workbook and a Parquet file, and look at what we get when we
read each one. Then we ask an API for data and look at the shapes its answer can take. Plan
about **50 minutes**.

You will:

- see what pandas reads from a CSV file, and why the types are worked out again at every read
- read an Excel workbook with more than one sheet, and find a column with two types in it
- write and read a Parquet file
- request data from an API and read its answer as JSON
- turn nested records into columns with `pd.json_normalize`, and lists inside records into
  rows with `record_path` and `explode`

One cell in Part 3 stops with an error on purpose, so run the cells one at a time rather
than all at once.

AI is welcome here. Ask it to explain a cell, or why your prediction was wrong.

In [1]:
import pandas as pd

## Part 1 · What pandas reads from a CSV file

You have loaded this file with `read_csv` many times. Before we load it again, look at what
the file holds. With the library `requests` you download it as text, and we print the first
lines:

In [2]:
import requests

url = "https://raw.githubusercontent.com/stat163-2026t1/week6-pre-lecture/main/data/online_retail_2010_11.csv"
text = requests.get(url).text
print(text[:2000])

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
529995,48184,DOORMAT ENGLISH ROSE ,6,2010-11-01 08:56:00,7.95,16316.0,United Kingdom
529995,48187,DOORMAT NEW ENGLAND,4,2010-11-01 08:56:00,7.95,16316.0,United Kingdom
529995,21523,DOORMAT FANCY FONT HOME SWEET HOME,10,2010-11-01 08:56:00,6.75,16316.0,United Kingdom
529995,22708,WRAP DOLLY GIRL,25,2010-11-01 08:56:00,0.42,16316.0,United Kingdom
529995,22781,GUMBALL MAGAZINE RACK,4,2010-11-01 08:56:00,7.65,16316.0,United Kingdom
529995,22899,CHILDREN'S APRON DOLLY GIRL ,6,2010-11-01 08:56:00,2.1,16316.0,United Kingdom
529995,22467,GUMBALL COAT RACK,12,2010-11-01 08:56:00,2.55,16316.0,United Kingdom
529995,22052,VINTAGE CARAVAN GIFT WRAP,25,2010-11-01 08:56:00,0.42,16316.0,United Kingdom
529995,22719,GUMBALL MONOCHROME COAT RACK,24,2010-11-01 08:56:00,1.25,16316.0,United Kingdom
529995,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-11-01 08:56:00,2.95,16316.0,United Kingdom
529995,71477,COLOUR GLASS. STAR T-LI

A CSV file is plain text. In the output you can see how it is laid out:

- The first line holds the column names.
- Every other line is one row, with commas between the values.
- Where a customer is missing, nothing stands between two commas: `0.0,,United Kingdom`.
- `Customer ID` is written with a dot: `16316.0`.

A value can hold commas of its own. Here is line 140 of the file, counting from 0 as Python
does:

In [3]:
text_lines = text.splitlines()
print(text_lines[140])

530003,22166,"WALL MIRROR , RECT DIAMANTE, PINK/",1,2010-11-01 09:14:00,16.13,,United Kingdom


The description `WALL MIRROR , RECT DIAMANTE, PINK/` has two commas in it, so the file puts
it in double quotes, and the commas inside the quotes belong to the value. Now load the file
as you always do, and look at that row. In the table it has the label 139, because the first
line of the file holds the column names:

In [4]:
df = pd.read_csv(url)
df.loc[139, "Description"]

'WALL MIRROR , RECT DIAMANTE, PINK/'

Every value reaches pandas as characters, and for each column pandas picks one type that
fits all of its values. You have seen what that gives: `StockCode` is text, because some
codes have a letter in them; `InvoiceDate` is text, because pandas does not guess dates; and
`Customer ID` is a float, because of the empty fields.

In [5]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

You have fixed the last two before:

In [6]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Customer ID"] = df["Customer ID"].astype("Int64")
df.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                   str
dtype: object

Now we write the fixed table to a new CSV file and read that file back. With
`index=False` you leave out the row labels, which would otherwise become a first column
without a name.

**Predict: which types will `InvoiceDate` and `Customer ID` have in `back`?**

In [ ]:
df.to_csv("retail.csv", index=False)
back = pd.read_csv("retail.csv")
back.dtypes

<details>
<summary>Answer</summary>

`InvoiceDate` is text again, and `Customer ID` is a float again. The types you set belonged
to the table in your notebook, and the file received only characters. The file now holds
`16316` without the `.0`, but the empty cells are still there, so pandas reads the column as
floats once more.

</details>

With a CSV file, the types are worked out again every time you read it. You can give them to
`read_csv` yourself:

In [8]:
back = pd.read_csv(
    "retail.csv", parse_dates=["InvoiceDate"], dtype={"Customer ID": "Int64"}
)
back.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                   str
dtype: object

With `sep` and `skiprows` you have told `read_csv` how a file is laid out. With
`parse_dates` and `dtype` you tell it what the columns hold: with `parse_dates` you name the
columns to read as dates, and with `dtype` you give a type to each column you name.

**Do this only with a file you have already profiled.** A type you set while reading has to
fit every value of the column. Remember the unemployment table from the last practice: its
`value` column held numbers, `:` for a missing value, and numbers with letters after them,
the flags. Turning that column into numbers stopped with an error, and with
`errors="coerce"` the flags became missing values. If you force a type while reading, the
read stops with that error before you have seen a single row. So load an unknown file as it
comes, profile it, and only then convert.

## Part 2 · Excel keeps a type in each cell

The online shop's data is published as an Excel workbook with one sheet per year. Our
workbook is smaller: two sheets, November 2010 and December 2010. In a workbook every cell
keeps a type: a number, a text or a date. The types travel with the file, so you do not work
them out again at every read. But the type belongs to the cell, so one column can hold values
of different types.

With `sheet_name=None` you read every sheet. You get a dictionary: the keys are the sheet
names, and each value is the table from that sheet.

In [9]:
excel_url = "https://raw.githubusercontent.com/stat163-2026t1/week6-pre-lecture/main/data/online_retail_2010.xlsx"
sheets = pd.read_excel(excel_url, sheet_name=None)
list(sheets)

['2010-11', '2010-12']

Without `sheet_name` you get the first sheet only, here November:

In [10]:
from_excel = pd.read_excel(excel_url)
from_excel.dtypes

Invoice                object
StockCode              object
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

`InvoiceDate` is a date with no help from you, because the workbook stores a date as a
date. From a workbook pandas does not guess a type from characters: it takes the type of each
cell, and it shows a column whose cells have more than one type as `object`. `StockCode`
holds codes such as `85123A`, so you would expect text.

**Predict: for how many rows will `.str.len()` give a missing value: none, some, or all?**

In [ ]:
from_excel["StockCode"].str.len().isna().sum()

<details>
<summary>Answer</summary>

Some: about nine rows in ten. The `.str` methods are for text: for a value that is not text,
the result is a missing value.

</details>

With `.map(type)` you apply Python's `type` to every value of the column, and with
`value_counts` you count the results:

In [12]:
from_excel["StockCode"].map(type).value_counts()

StockCode
<class 'int'>    69026
<class 'str'>     8989
Name: count, dtype: int64

In the workbook, a code made only of digits, such as `85123`, is a number, and a code with
a letter in it, such as `85123A`, is text. One column, two types. In the CSV file both
were characters, and pandas read the whole column as text. Here you make it text yourself:

In [13]:
from_excel["StockCode"] = from_excel["StockCode"].astype(str)
from_excel["StockCode"].str.len().isna().sum()

np.int64(0)

**Try it.** Count the types of the values in `Invoice`.

In [14]:
# Try it: count the types of the values in from_excel["Invoice"]

## Part 3 · Parquet keeps one type per column

**Parquet** is a file format made for tables. It stores every column with one type for the
whole column, so the table you read is the table you wrote. It is a binary format: in a text
editor you would see unreadable symbols, and you read it only with a program such as pandas.

In [15]:
df.to_parquet("retail.parquet")
back = pd.read_parquet("retail.parquet")
back.dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             Int64
Country                   str
dtype: object

The dates and the `Int64` column are as you wrote them, and you passed no arguments.

Because a Parquet column has one type, a column with two types cannot go into the file. In
`from_excel`, `Invoice` still holds numbers and text:

In [16]:
from_excel.to_parquet("from_excel.parquet")

ArrowInvalid: ("Could not convert 'C529999' with type str: tried to convert to int64", 'Conversion failed for column Invoice with type object')

pandas stops with an error. The first values of `Invoice` are numbers, so pandas started a
column of whole numbers, and `C529999`, a cancelled invoice, does not fit in it. A workbook
takes the mixed column with no error. For Parquet you have to decide on one type before you
save. An invoice number such as `C529999` cannot be a number, so make the column
text, as you did with `StockCode`:

In [17]:
from_excel["Invoice"] = from_excel["Invoice"].astype(str)
from_excel.to_parquet("from_excel.parquet")
pd.read_parquet("from_excel.parquet").dtypes

Invoice                   str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

The file is written, and every column comes back with one type.

**Predict: `retail.csv` from Part 1 and `retail.parquet` hold the same table. Which file is
smaller?**

In [ ]:
import os

print(os.path.getsize("retail.csv"))
print(os.path.getsize("retail.parquet"))

<details>
<summary>Answer</summary>

The Parquet file is about twelve times smaller. A CSV file holds every value written out in
full, on every line: `United Kingdom` appears about 72,000 times. In a Parquet file the values
of one column sit together, compressed, and a column that repeats a few values thousands of
times takes almost no space. Numbers and dates are stored in binary form, as they are in
memory.

</details>

Because each column is stored separately, you can read only the columns you need. From a CSV
file pandas has to read every line in full, whichever columns you keep.

In [19]:
pd.read_parquet("retail.parquet", columns=["Country", "Quantity"]).head()

,Country,Quantity
0,United Kingdom,6
1,United Kingdom,4
2,United Kingdom,10
3,United Kingdom,25
4,United Kingdom,4


A CSV file suits a table you send to someone else: any program opens it, and a person can
read it. Parquet suits a table you keep for your own analysis in code.

## Part 4 · Asking an API for data

An **API** is an address on the web that answers with data instead of a page for people to
read. You send a request, with parameters that say what you want, and the answer arrives
as text.

For this course we run a small API. It serves the orders of a café over two weeks, made up
and kept small so that they are easy to read. You send requests to it with `requests.get`,
as you did for the CSV file in Part 1. Its first page says what it offers:

In [20]:
api = "https://stat163-data.oleh-omelchenko.workers.dev"
response = requests.get(api)
print(response.status_code)
print(response.text[:300])

200
{"about":"A made-up café's orders, 14-27 September 2026. Closed on Mondays.","/cafe/orders":{"what":"One record per order, with its items inside.","parameters":{"date":"only the orders of one day, such as 2026-09-15","guest":"only the orders of one loyalty card, such as 1042","page":"which page of t


The status code 200 means that the request worked. The text is **JSON**, written with the
brackets you know from Python: `{}` around a record of named fields, `[]` around a list, and
quotes around text. Numbers have no quotes. With `.json()` you turn the text into Python
lists and dictionaries:

In [21]:
response.json()

{'about': "A made-up café's orders, 14-27 September 2026. Closed on Mondays.",
 '/cafe/orders': {'what': 'One record per order, with its items inside.',
  'parameters': {'date': 'only the orders of one day, such as 2026-09-15',
   'guest': 'only the orders of one loyalty card, such as 1042',
   'page': 'which page of the answer, from 1',
   'per_page': 'records per page, 10 unless you ask, at most 200'}},
 '/cafe/orders/{id}': 'One order, by its id.',
 '/cafe/days': "One record per day, with the ids of that day's orders."}

The API lists its addresses and the parameters each one takes. In `/cafe/orders/{id}`, the
`{id}` stands for the number of one order, as in `/cafe/orders/2`. We ask `/cafe/orders` for the
orders of one guest, the one with loyalty card 1042. With `params` you name the parameters,
and they are added to the address for you.

In [22]:
response = requests.get(api + "/cafe/orders", params={"guest": 1042})
data = response.json()
data

{'page': 1,
 'pages': 1,
 'per_page': 10,
 'total': 3,
 'orders': [{'id': 2,
   'date': '2026-09-15',
   'time': '08:40',
   'table': 4,
   'guest': {'name': 'Olena', 'card': 1042},
   'items': [{'dish': 'cappuccino',
     'qty': 2,
     'price': 70,
     'extras': ['oat milk']},
    {'dish': 'croissant', 'qty': 1, 'price': 55, 'extras': []}]},
  {'id': 125,
   'date': '2026-09-17',
   'time': '13:05',
   'table': 6,
   'guest': {'name': 'Olena', 'card': 1042},
   'items': [{'dish': 'soup', 'qty': 1, 'price': 95, 'extras': ['bread']}]},
  {'id': 315,
   'date': '2026-09-22',
   'time': '08:35',
   'table': 7,
   'guest': {'name': 'Olena', 'card': 1042},
   'items': [{'dish': 'cappuccino',
     'qty': 1,
     'price': 70,
     'extras': ['oat milk', 'syrup']}]}]}

The answer is a record. `orders` holds the records you asked for, and the other fields
describe the answer: `total` is the number of records that match, here three.

Now ask for one whole day:

In [23]:
response = requests.get(api + "/cafe/orders", params={"date": "2026-09-15"})
data = response.json()
print(data["total"])
print(data["per_page"])
print(data["pages"])

56
10
6


**Predict: read `total` and `per_page`. How many orders will `data["orders"]` hold?**

In [ ]:
len(data["orders"])

<details>
<summary>Answer</summary>

10 of the 56. The answer comes in **pages** of 10 records, and `pages` says there are six of
them.

</details>

Answers in pages are common: each time, compare the number of records you received with the
total in the answer.

## Part 5 · Records and lists inside records

Back to the three orders of guest 1042:

In [25]:
small = requests.get(api + "/cafe/orders", params={"guest": 1042}).json()["orders"]
small

[{'id': 2,
  'date': '2026-09-15',
  'time': '08:40',
  'table': 4,
  'guest': {'name': 'Olena', 'card': 1042},
  'items': [{'dish': 'cappuccino',
    'qty': 2,
    'price': 70,
    'extras': ['oat milk']},
   {'dish': 'croissant', 'qty': 1, 'price': 55, 'extras': []}]},
 {'id': 125,
  'date': '2026-09-17',
  'time': '13:05',
  'table': 6,
  'guest': {'name': 'Olena', 'card': 1042},
  'items': [{'dish': 'soup', 'qty': 1, 'price': 95, 'extras': ['bread']}]},
 {'id': 315,
  'date': '2026-09-22',
  'time': '08:35',
  'table': 7,
  'guest': {'name': 'Olena', 'card': 1042},
  'items': [{'dish': 'cappuccino',
    'qty': 1,
    'price': 70,
    'extras': ['oat milk', 'syrup']}]}]

Each order holds more data inside it:

- `guest` holds a record of its own, with a `name` and a `card`. A record inside a record is
  a **nested record**.
- `items` holds a **list** of records, one for each dish. The first order has two items, the
  other two have one.
- Each item has `extras`, a list of plain values: words, not records. The croissant's list is
  empty, `[]`.

To reach a value inside, you go one step down for each level:

In [26]:
small[0]["items"][0]["extras"][0]

'oat milk'

With `pd.DataFrame` you get one row per order, and each cell of `guest` and `items` holds the
whole dictionary or list:

In [27]:
pd.DataFrame(small)

,id,date,time,table,guest,items
0,2,2026-09-15,08:40,4,"{'name': 'Olena', 'card': 1042}","[{'dish': 'cappuccino', 'qty': 2, 'price': 70,..."
1,125,2026-09-17,13:05,6,"{'name': 'Olena', 'card': 1042}","[{'dish': 'soup', 'qty': 1, 'price': 95, 'extr..."
2,315,2026-09-22,08:35,7,"{'name': 'Olena', 'card': 1042}","[{'dish': 'cappuccino', 'qty': 1, 'price': 70,..."


You cannot sort or group by such a column.

With `pd.json_normalize` you give every field of a nested record a column of its own. The
name of the new column joins the field names with a dot.

**Predict: `guest` gets two columns. What will `json_normalize` do with `items`?**

In [ ]:
pd.json_normalize(small)

<details>
<summary>Answer</summary>

`guest` became two columns, `guest.name` and `guest.card`. `items` stayed as it was: each cell
holds the whole list. With `json_normalize` you open records, but not lists. An order can
have one item or ten, so its items cannot become a fixed set of columns. Each item needs a
row of its own.

</details>

With `record_path` you name the list, and you get a row for each item of it. With `meta` you
name fields of the record that holds the list, and they are copied onto those rows, so every
item keeps the order it came from.

**Predict: how many rows will `items` have, and what will happen to `extras`?**

In [ ]:
items = pd.json_normalize(small, record_path="items", meta=["id", "time"])
items

<details>
<summary>Answer</summary>

Four rows, one for each item: two, one and one. `extras` stayed a list: with `record_path` you
open the list you name, and every other list stays as it is.

</details>

One row of `items` is one dish in one order. One row of `small` was one order, so the table
has a new grain.

A list of plain values has no fields to open, so you turn it into rows with `explode`: each
value of the list gets a row of its own, and the other columns are copied onto it.

**Predict: how many rows will `with_extras` have?**

In [ ]:
with_extras = items.explode("extras")
with_extras

<details>
<summary>Answer</summary>

Five. The last cappuccino has two extras, so it has two rows, and the row label `3` appears
twice. The croissant's list is empty: with `explode` it becomes one row with a missing value,
so the croissant stays in the table.

</details>

In JSON, records and lists hold other data, and they can sit inside each other. Look at what
sits inside each record, and choose what one row of your table should be:

| Inside a record | What you do | One row of the table |
|---|---|---|
| a record | `json_normalize`: a column per field, such as `guest.name` | stays the same |
| a list of records | `record_path` with `meta`: a row per item | stands for one item of the list |
| a list of plain values | `explode`: a row per value | stands for one value of the list |

**Try it.** Ask the API for the orders of loyalty card 1077 on 16 September: give both
`guest` and `date` in `params`. Check `total` against the number of orders you received,
then turn their items into rows.

In [31]:
# Try it: the orders of card 1077 on 2026-09-16, one row per item